In [1]:
import os
import re
import fitz  # PyMuPDF for PDF text extraction
import pandas as pd
from langdetect import detect
import fitz
import os
import pandas as pd
import re
import signal
from langdetect import detect  # Import langdetect
import signal
import re
from docx import Document
import fitz


start_keyword = [
    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(PROGRAM\s*SUMMARY)\b\s*(?:\n|$)",
    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(PROJECT\s*SUMMARY)\b\s*(?:\n|$)",
    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(EXECUTIVE\s*SUMMARY)\b\s*(?:\n|$)",
    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(TC\s*Document)\b\s*(?:\n|$)",
    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(Technical\s*Cooperation\s*Document)\b\s*(?:\n|$)",
    r"S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y",  # "SUMMARY"
    r"P\s{1}R\s{1}O\s{1}J\s{1}E\s{1}C\s{1}T\s{1}S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y",  # "PROJECTS SUMMARY"
    r"\bSUMMARY\s{1}T\s{1}A\s{1}B\s{1}L\s{1}E\b",  # "SUMMARY TABLE"
    r"Summary\s{1}o\s{1}f\s{1}P\s{1}R\s{1}O\s{1}P\s{1}O\s{1}S\s{1}A\s{1}L",  # "Summary of Proposal"
    r"R\s{1}E\s{1}Q\s{1}U\s{1}E\s{1}S\s{1}T\s{1}S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y",  # "REQUESTS SUMMARY"
    r"T\s{1}E\s{1}C\s{1}H\s{1}N\s{1}I\s{1}C\s{1}A\s{1}L\s{1}A\s{1}S\s{1}S\s{1}I\s{1}S\s{1}T\s{1}A\s{1}N\s{1}C\s{1}E\s{1}R\s{1}E\s{1}Q\s{1}U\s{1}E\s{1}S\s{1}T\s{1}S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y",  # "TECHNICAL ASSISTANCE REQUESTS SUMMARY"
    r"\bP\s{1}R\s{1}O\s{1}G\s{1}R\s{1}A\s{1}M\s{1}M\s{1}E\s{1}S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y\b",  # "PROGRAMMES SUMMARY"
    r"\bP\s{1}R\s{1}O\s{1}G\s{1}R\s{1}A\s{1}M\s{1}S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y\b",  # "PROGRAMS SUMMARY"
    r"\bP\s{1}R\s{1}O\s{1}G\s{1}R\s{1}A\s{1}M\s{1}M\s{1}S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y\b",  # "PROGRAMMES SUMMARY"
    r"\bP\s{1}R\s{1}O\s{1}J\s{1}E\s{1}C\s{1}T\s{1}S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y\b",  # "PROJECTS SUMMARY"
    r"\bE\s{1}X\s{1}E\s{1}C\s{1}U\s{1}T\s{1}I\s{1}V\s{1}E\s{1}S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y\b",  # "EXECUTIVE SUMMARY"
    r"\bS\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y\s{1}T\s{1}A\s{1}B\s{1}L\s{1}E\b",  # "SUMMARY TABLE"
    r"S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y\s{1}O\s{1}F\s{1}P\s{1}R\s{1}O\s{1}P\s{1}O\s{1}S\s{1}A\s{1}L",  # "SUMMARY OF PROPOSAL"
    r"R\s{1}E\s{1}Q\s{1}U\s{1}E\s{1}S\s{1}T\s{1}S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y",  # "REQUESTS SUMMARY"
    r"T\s{1}E\s{1}C\s{1}H\s{1}N\s{1}I\s{1}C\s{1}A\s{1}L\s{1}A\s{1}S\s{1}S\s{1}I\s{1}S\s{1}T\s{1}A\s{1}N\s{1}C\s{1}E\s{1}R\s{1}E\s{1}Q\s{1}U\s{1}E\s{1}S\s{1}T\s{1}S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y",  # "TECHNICAL ASSISTANCE REQUESTS SUMMARY"
    r"\bP\s{1}R\s{1}O\s{1}G\s{1}R\s{1}A\s{1}M\s{1}M\s{1}E\s{1}S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y\b",  # "PROGRAMMES SUMMARY"
    r"\bP\s{1}R\s{1}O\s{1}G\s{1}R\s{1}A\s{1}M\s{1}S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y\b",  # "PROGRAMS SUMMARY"
    r"\bP\s{1}R\s{1}O\s{1}G\s{1}R\s{1}A\s{1}M\s{1}M\s{1}S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y\b",  # "PROGRAMMES SUMMARY"
    r"\bP\s{1}R\s{1}O\s{1}J\s{1}E\s{1}C\s{1}T\s{1}S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y\b",  # "PROJECTS SUMMARY"
    r"\bE\s{1}X\s{1}E\s{1}C\s{1}U\s{1}T\s{1}I\s{1}V\s{1}E\s{1}S\s{1}U\s{1}M\s{1}M\s{1}A\s{1}R\s{1}Y\b",  
    r"Basic\s*Information\s*for\s*TC",
    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(PROJECT\s*ABSTRACT)\b\s*(?:\n|$)",
    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b\b(SUMMARY TABLE|PROGRAMME SUMMARY|PROGRAM SUMMARY|PROGRAMM SUMMARY|PROJECT SUMMARY|EXECUTIVE SUMMARY)\b",
    r"\n{1,}\b(SUMMARY TABLE|PROGRAMME SUMMARY|PROGRAM SUMMARY|PROGRAMM SUMMARY|PROJECT SUMMARY|EXECUTIVE SUMMARY)\b",
    r"\n{1,}(Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|STUDY|Executive)\s*SUMMARY",
    r"^(?:\S+\s+){0,3}Summary(?:\s+\S+){0,3}$",  # Retain this if "Summary" as a heading is important
    r"(?i)SUMMARY TABLE",  # "SUMMARY TABLE"
    r"SUMMARY TABLE",
    r"\n{1,}\s*SUMMARY(\s{2,}|(\s*\n))",
    r"SUMMARY OF THE PROPOSED MODIFICATION",
    r"(Project’s|PROGRAM|PROGRAMM|PROGRAMME|PROJECT|STUDY|EXECUTIVE)\s*SUMMARY(\s{3,}|\n)",
    r"\s{2,}(Project’s|PROGRAM|PROGRAMM|PROGRAMME|PROJECT|STUDY|EXECUTIVE)\s*SUMMARY(\s{3,}|\n)",
    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b((Project’s|PROGRAM|PROGRAMM|PROGRAMME|Project|STUDY|Executive)\s*SUMMARY)\b\s*(?:\n|$)",
    r"PROJECT\s*SUMMARY(\s{3,}|(\s*\n))",
    r"^(?:\S+\s+){0,3}Summary(?:\s+\S+){0,3}$",
    r"(?:\S+\s+){0,3}Summary(?:\s+\S+){0,3}$",
    r"MEMORANDUM OF ASSISTANCE FOR PROJECT PREPARATION AND EXECUTION",
    r"Memorandum of Assistance for Project"
    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(PROJECT\s*ABSTRACT)\b\s*(?:\n|$)",
    r"PROJECT\s*ABSTRACT"
    
]

end_keyword = [
    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(ANNEXES)\b\s*(?:\n|$)",
    r"\b[IVIX]{1,3}\s*[-\.)]?\s*PROBLEM\s*TO\s*BE\s*ADDRESSED",
    r"\b[IVIX]{1,3}\s*[-\.)]?\s*(PROJECT|PROJECTS|PROGRAM)\s*DESCRIPTION\s*AND\s*RESULTS\s*MONITORING",
    r"\b[IVIX]{1,3}\s*[-\.)]?\s*DESCRIPTION\s*AND\s*RESULTS\s*MONITORING",
    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*(PROJECT|PROJECTS|PROGRAM)\s*DESCRIPTION\s*AND\s*RESULTS\s*MONITORING(\s{3,}|\n)", #### Perfect pattern for numbers (1.1.1) / 1 / 1.1
    r"\b[B-F]\s*[-\.)]?\s*\n?\s*(PROJECT|PROJECTS|PROGRAM)\s*DESCRIPTION\s*AND\s*RESULTS\s*MONITORING", #### for starting from letters
    r"\b[IVIX]{1,3}\s*[-\.)]?\s*\n?\s*(PROJECT|PROJECTS|PROGRAM)\s*DESCRIPTION\s*AND\s*RESULTS\s*MONITORING", ### Roman starting (1-9)
    r"\d{1}\s*[-)\.]\s*n?\s*(PROJECT|PROJECTS|PROGRAM)\s*DESCRIPTION\s*AND\s*RESULTS\s*MONITORING", ### Single Digit
    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*DESCRIPTION\s*AND\s*RESULTS\s*MONITORING(\s{3,}|\n)", #### Perfect pattern for numbers (1.1.1) / 1 / 1.1
    r"\b[B-F]\s*[-\.)]?\s*\n?\s*DESCRIPTION\s*AND\s*RESULTS\s*MONITORING", #### for starting from letters
    r"\b[IVIX]{1,3}\s*[-\.)]?\s*\n?\s*DESCRIPTION\s*AND\s*RESULTS\s*MONITORING", ### Roman starting (1-9)
    r"\d{1}\s*[-)\.]\s*n?\s*DESCRIPTION\s*AND\s*RESULTS\s*MONITORING", ### Single Digit
    r"\b[IVIX]{1,3}\s*[-\.)]?\s*FRAME\s*OF\s*REFERENCE",
    r"\b[IVIX]{1,3}\s*[-\.)]?\s*COUNTRY\s*AND\s*PROJECT\s*ELIGIBILITY",
    r"\b[IVIX]{1,3}\s*[-\.)]?\s*BACKGROUND\s*AND\s*RATIONALE",
    r"\b[IVIX]{1,3}\s*[-\.)]?\s*Objectives\s*and\s*Justification\s*of\s*the\s*TC",
    r"\b[IVIX]{1,3}\s*[-\.)]?\s*COUNTRY\s*ELIGIBILITY",
    r"\b[IVIX]{1,3}\s*[-\.)]?\s*DESCRIPTION\s*OF\s*THE\s*EVENT",
    r"\b[IVIX]{1,3}\s*[-\.)]?\s*DESCRIPTION\s*OF",#this is bad
    r"Objectives\s*and\s*Justification\s*of\s*the\s*TC",
    r"\b[IVIX]{1,3}\s*[-\.)]?\s*Background",
    r"LIST OF ACRONYMS",  # "LIST OF ACRONYMS"
    r"ACRONYMS",  # "ACRONYMS"
    r"T\s*a\s*b\s*l\s*e\s*\s*o\s*f\s*\s*C\s*o\s*n\s*t\s*e\s*n\s*t",
    r"\bTable of Contents\b",  # "Table of Contents"
    r"T\s{1}A\s{1}B\s{1}L\s{1}E",
    r"F\s{1}R\s{1}A\s{1}M\s{1}E\s{1}W\s{1}O\s{1}R\s{1}K",  # "FRAMEWORK"
    r"R\s{1}E\s{1}S\s{1}U\s{1}L\s{1}T",  # "RESULT"
    r"L\s{1}O\s{1}G\s{1}I\s{1}C\s{1}A\s{1}L",  # "LOGICAL"
    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})ANNEX",
    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})(LOGICAL|Result|Results)\s*FRAMEWORK",
    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})(LOGICAL|Result|Results)\s*Matrix",
    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})DESCRIPTION OF THE PROGRAM"
       
]



In [2]:
def is_pdf_in_english(text):
    try:
        return detect(text) == 'en'
    except Exception as e:
        print(f"Error detecting language: {e}")
        return False

# Define a custom exception for timeout
class TimeoutException(Exception):
    pass

# Timeout handler
def timeout_handler(signum, frame):
    raise TimeoutException()
    
def remove_illegal_characters(df):
    # Define a function to clean individual cells
    def clean_cell(value):
        if isinstance(value, str):
            # Remove illegal characters but preserve line spaces, tabs, and carriage returns
            return re.sub(r'[^\x09\x0A\x0D\x20-\x7E]', '', value)
        return value

    # Apply the cleaning function to the entire DataFrame
    return df.applymap(clean_cell)

        



def clean_text(text, cleaning_patterns):
    # Loop through the list of regex patterns
    for pattern in cleaning_patterns:
        # Search for the first occurrence of the pattern in the text
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            # Print the pattern used for truncation
            print(f"Pattern used to truncate text:\n\n----------- {pattern}")
            
            # Truncate the text after the first occurrence of the matched pattern
            text = text[:match.start()]
            
            # Break after truncation
            break
    
    return text
        
    



def clean_summary(text, cleaning_patterns):
    for pattern in cleaning_patterns:
        #print(pattern)
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            print(f"Pattern found: {pattern}")
            text = text[:match.end()]  # Truncate after the matched pattern

    return text

def clean_more(text, cleaning_patterns):
    for pattern in cleaning_patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            print(f"Pattern found: {pattern}")
            # Remove the matched pattern and truncate the text
            text = text[:match.start()]  # Keep only the part before the matched pattern

    return text


def identify_table_of_contents(pdf):
    """
    Identifies the Table of Contents (TOC) page based on specific patterns.
    Returns the page number (1-indexed) or 0 if not found.
    """
    # List of patterns to match variations of "Table of Contents" and "Contents"
    toc_patterns = [
        r"T\s{1,}a\s{1,}b\s{1,}l\s{1,}e\s{1,}o\s{1,}f\s{1,}\s{1,}C\s{1,}o\s{1,}n\s{1,}t\s{1,}e\s{1,}n\s{1,}t\s{1,}s",  # Spaced 'Table of Contents'
        r"C\s{1,}o\s{1,}n\s{1,}t\s{1,}e\s{1,}n\s{1,}t\s{1,}s",  # Spaced 'Contents'
        r"T A B L E O F C O N T E N T S",  # Uppercase 'TABLE OF CONTENTS'
        r"(\n{1,}|^|\s*\n\s*|\s{2,})\b(Table\s*of\s*Contents)\b\s*(?:\n|$)"
  # 'Table of Contents' with optional spaces
        r"(\n{1,}|^|\s*\n\s*|\s{2,})\bContents\b\s*(?:\n|$)",  # Standalone 'Contents'
        r"(\n{1,}|^|\s*\n\s*|\s{2,})\bCONTENT\b\s*(?:\n|$)",  # Standalone 'CONTENT'
        r"Table\s*of\s*Contents",  # Simple 'Table of Contents'
        r"Contents"  # Simple 'Contents'
    ]
    
    # Iterate over each page in the PDF
    for page_num in range(pdf.page_count):
        page = pdf[page_num]
        page_text = page.get_text("text")
        
        # Check each pattern
        for pattern in toc_patterns:
            if re.search(pattern, page_text, re.IGNORECASE):
                print(f"Matched Pattern: {pattern}")  # Print the matched pattern
                return page_num + 1  # Return only the page number
    
    # Default to the first page if no TOC is found
    return 0

In [5]:


def extract_summary_section(file_path):
    """
    Extracts the summary section from the PDF based on start and end keywords,
    skipping only the Table of Contents (TOC) page and limiting the extracted content to two pages if no end pattern is found.
    """
    try:
        pdf_document = fitz.open(file_path)

        # Identify the TOC page
        toc_page = identify_table_of_contents(pdf_document) - 1  # Convert to 0-indexed
        print(f"Table of Contents identified on page: {toc_page + 1}")

        # Combine all pages into a single string, skipping only the TOC page
        filtered_pages = []
        for page_num in range(pdf_document.page_count):
            if page_num == toc_page:  # Skip only the TOC page
                continue
            page_text = pdf_document[page_num].get_text("text")
            filtered_pages.append(page_text)

        filtered_content = "\f".join(filtered_pages)

        # Find the start match
        start_index, matched_start_pattern, start_page_index, start_line = None, None, None, None
        for pattern in start_keyword:
            match = re.search(pattern, filtered_content, re.DOTALL | re.IGNORECASE)
            if match:
                start_index = match.end()
                matched_start_pattern = pattern
                try:
                    start_page_index = next(
                        (i for i, page in enumerate(filtered_pages) if match.group(0) in page), None
                    )
                except StopIteration:
                    start_page_index = None

                if start_page_index is not None:
                    page_lines = filtered_pages[start_page_index].splitlines()
                    for line in page_lines:
                        if re.search(pattern, line, re.IGNORECASE):
                            start_line = line.strip()
                            break

                print(f"Matched start pattern: {matched_start_pattern} on page {start_page_index + 1 if start_page_index is not None else 'Unknown'}")
                print(f"Actual line containing start pattern: {start_line}")
                break

        if start_index is None:
            print("Start Summary Table not found.")
            pdf_document.close()
            return "", ""

        # Find the first occurrence of any end pattern
        earliest_end_index, matched_end_pattern = None, None
        for pattern in end_keyword:
            match = re.search(pattern, filtered_content[start_index:], re.DOTALL | re.IGNORECASE)
            if match:
                candidate_end_index = start_index + match.start()
                if earliest_end_index is None or candidate_end_index < earliest_end_index:
                    earliest_end_index = candidate_end_index
                    matched_end_pattern = pattern

        if earliest_end_index is not None:
            try:
                end_page_index = next(
                    (i for i, page in enumerate(filtered_pages) if re.search(matched_end_pattern, page, re.IGNORECASE)), 
                    None
                )
            except StopIteration:
                end_page_index = None

            print(f"Matched end pattern: {matched_end_pattern} on page {end_page_index + 1 if end_page_index is not None else 'Unknown'}")

        # Determine the end index
        if earliest_end_index is not None:
            end_index = earliest_end_index
        elif start_page_index is not None:
            two_page_limit_index = len("\f".join(filtered_pages[: start_page_index + 2]))
            end_index = two_page_limit_index
        else:
            end_index = len(filtered_content)

        pdf_document.close()

        if start_index is not None and end_index is not None:
            extracted_text = filtered_content[start_index:end_index].strip()

            # Additional cleaning: Remove all end patterns from extracted text
            for pattern in end_keyword:
                extracted_text = re.sub(pattern, '', extracted_text, flags=re.IGNORECASE)

            # Remove extra blank lines and whitespace
            extracted_text = re.sub(r'\n{2,}', '\n', extracted_text).strip()

            # Count the number of extracted pages
            extracted_page_count = extracted_text.count("\f") + 1  # Add 1 since \f separates pages

            print("Summary Table found!!!")
            print(f"Extracted text spans {extracted_page_count} pages.")

            # Call clean_text if the extracted text spans more than 4 pages
            if extracted_page_count > 4:
                print("Extracted text exceeds 4 pages, applying additional cleaning...")
                extracted_text = clean_summary(extracted_text, end_keyword)

           
            return extracted_text, matched_start_pattern
        else:
            print("Could not determine the end of the Summary Table.")
            return "", ""

    except Exception as e:
        print(f"An error occurred: {e}")
        return "Error processing the PDF", ""




        
def process_pdfs_in_folder(file_path, output_excel):
    results = []
    filename = os.path.basename(file_path)
    
    if filename.endswith(('.pdf', '.PDF')):
        print(f"\nProcessing file: {filename}")
        try:
            pdf_document = fitz.open(file_path)

            # Extract all text for language check
            text = ""
            for page_num in range(pdf_document.page_count):
                text += pdf_document[page_num].get_text("text")
                
              

                # Check language first
                if not is_pdf_in_english(text):
                    print("File not in english")
                    results.append([filename, "Not Supported", "Not Supported"])
                    pdf_document.close()
                    continue
                    
                    # Define your start and stop patterns for extracting components
                project_component_start = [
                    r"\b[A-F]\s*[-\.)]?\s*Project\s*(components|component)([\s\S]*)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[IVIX]{1,3}\s*[-\.)]?\s*Project\s*(components|component)([\s\S]*)",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*(Objectives|Objective),\s*components(,?)\s*and\s*cost(\s{3,}|\n)([\s\S]*)",  # Perfect pattern for numbers (1.1.1) / 1 / 1.1
                    r"\b[A-F]\s*[-\.)]?\s*(Objectives|Objective),\s*components,\s*and\s*cost([\s\S]*)",  # for starting from letters
                    r"\b[A-F]\s*[-\.)]?\s*(Objectives|Objective),\s+components(,?)\s+and\s+cost([\s\S]*)",
                    r"\b[IVIX]{1,3}\s*[-\.)]?\s*\n?\s*(Objectives|Objective),\s*components(,?)\s*and\s*cost([\s\S]*)",  # Roman starting (1-9)
                    r"\d{1}\s*[-)\.]\s*n?\s*(Objectives|Objective),\s*components(,?)\s*and\s*cost([\s\S]*)",  # Single Digit
                    r"\b[IVIX]{1,3}\s*[-\.)]?\s*Program\s*Description\s*and\s*Budget([\s\S]*)",  # Handles spacing variations with "PROGRAMME COMPONENTS"
                    r"\b[A-F]\s*[-\.)]?\s*Objectives\s*and\s*components([\s\S]*)",
                    r"\b[A-F]\s*[-\.)]?\s*Program\s*Objectives\s*and\s*Components([\s\S]*)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})(?:\n|$)\b(Objectives\s*and\s*components)\b(?:\n|$)([\s\S]*)",
                    r"\b[IVIX]{1,3}\s*[-\.)]?\s*\n?\s*Description\s*of\s*activities/components\s*and\s*budget([\s\S]*)",
                    r"\b[A-F]\s*[-\.)]?\s*Description\s*of\s*activities/components\s*and\s*budget([\s\S]*)",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.]?\s*\n?\s*Description\s*of\s*activities/components\s*and\s*budget([\s\S]*)",
                    r"\b[A-F]\s*[-\.)]?\s*\b(Objectives|Objective)\s*and\s*components([\s\S]*)",
                    r"(OBJECTIVE|OBJECTIVES),\s*EXPECTED\s*(RESULTS|RESULT),\s*(COMPONENTS|COMPONENT),\s*AND\s*(COST|COSTS)([\s\S]*)",
                    r"\b[A-F]\s*[-\.)]?\s*\bStructure\b\s*(?:\n|$)([\s\S]*)",
                    r"\b[A-F]\s*[-\.)]?\s*(Program|Project)\s*structure\s*(?:\n|$)([\s\S]*)",
                    r"\b[A-F]\s*[-\.)]?\s*Objective\s*(?:\n|$)([\s\S]*)",
                    r"\b[A-F]\s*[-\.)]?\s*Description\s*(?:\n|$)([\s\S]*)",
                    r"\b[A-F]\s*[-\.)]?\s*\b(components)\b\s*(?:\n|$)([\s\S]*)",
                    r"\b[A-F]\s*[-\.)]?\s*\b(components)\b\s*(?:\n|$)([\s\S]*)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\bComponents\b\s*(?:\n|$)([\s\S]*)",
                    r"\b[A-F]\s*[-\.)]?\s*Objectives\s*and\s*description([\s\S]*)",
                    r"Objectives, component, and cost of the modification([\s\S]*)",
                    r"(Objectives|Objective),\s*components(,?)\s*and\s*cost([\s\S]*)",
                    r"(Objectives|Objective)\s*and\s*components([\s\S]*)",
                    r"\b[A-F]\s*[-\.)]?\s*Key\s*project\s*infrastructure\s*components\s*and\s*schedule([\s\S]*)",
                    r"Objectives\s*and\s*description([\s\S]*)",
                    r"Program\s*Description\s*and\s*Budget([\s\S]*)",
                    r"Project Description and Environmental Setting([\s\S]*)",
                    r"main\s*components:([\s\S]*)",
                    r"components:\s*([\s\S]*)",
                    r"three\s*main\s*components([\s\S]*)",
                    r"three\s*components:([\s\S]*)", #16
                    r"two\s*components:([\s\S]*)", 
                    r"four\s*components:([\s\S]*)",
                    r"five\s*components:([\s\S]*)",
                    r"\d+(\.\d+)*\s*\n?\s*Component\s+I\s*:([\s\S]*)",
                    r"Component\s*I\s*:([\s\S]*)",
                    r"Component\s+1\s*(.|,)([\s\S]*)",
                    r"Component\s+I\s+([\s\S]*)",
                    r"Component\s+I\s*:([\s\S]*)",
                    r"Component\s+(I|1|One|A)\s+([\s\S]*)",
                    r"following\s*components:([\s\S]*)"
                ]

                project_component_end = [
                    r"\b[IVIX]{1,3}\s*[-\.)]?\s*COST\s*ESTIMATES\s*AND\s*FINANCING\s*PLAN",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*Project\s*administration,\s*evaluation,\s*and\s*auditing(\s{3,}|\n)",  # Perfect pattern for numbers (1.1.1) / 1 / 1.1
                    r"\b[B-F]\s*[-\.)]?\s*Project\s*administration,\s*evaluation,\s*and\s*auditing",  # for starting from letters
                    r"\b[B-F]\s*[-\.)]?\s*(Results|Result)\s*Matrix\s*with\s*indicators",  # for starting from letters
                    r"\b[B-F]\s*[-\.)]?\s*Key\s+results\s+indicators",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Key\s+results\s+indicators",
                    r"\b[B-F]\s*[-\.)]?\s+Key\s+results\s+indicators",
                    r"\b[B-F]\s*[-\.)]?\s+Key\s+indicators",
                    r"\b[B-F]\s*[-\.)]?\s*Key\s+results\s+indicators\b",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})(?:\n|$)\b(Key\s+results\s+indicators)\b",
                    r"\b[B-F]\s*[-\.)]?\s*Key\s*Result\s*Indicators",
                    r"\b[B-F]\s*[-\.)]?\s*Expected\s+outcomes\s+of\s+the\s+PROPEF",
                    r"\b[B-F]\s*[-\.)]?\s*Expected\s+outcomes",
                    r"\b[B-F]\s*[-\.)]?\s*Costs\s*and\s*Financing",
                    r"\b[B-F]\s*[-\.)]?\s*Expected\s*results\s*and\s*effectiveness\s*framework",
                    r"\b[B-F]\s*[-\.)]?\s*Financing\s*and\s*cost",
                    r"\b[B-F]\s*[-\.)]?\s*Costs\s*and\s*sources\s*of\s*financing",
                    r"\b[B-F]\s*[-\.)]?\s*Strategic alignment",
                    r"\b[IVIX]{1,3}\s*[-\.)]?\s*Execution agency and execution structure",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[IVIX]{1,3}\s*[-\.)]?\s*(Costs|Cost)\s*and\s*Financing",
                    r"\b[IVIX]{1,3}\s*[-\.)]?\s*COST\s*AND\s*FINANCING",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[B-F]\s*[-\.)]?\s*Key\s+results\s+indicators",
                    r"^[\s]*[B-F]\s*[-\.)]?\s*Key\s+results\s+indicators$",
                    r"\b[B-F]\s*[-\.)]?\s*Costs\s*and\s*Financing",
                    r"((\b[A-F])|(\b[IVIX]{1,3})|(\b\d{1,2}(\.\d{1,2}){0,2})|(\d{1}))\s*[-\.)]?\s*\n?\s*Key\s*results\s*indicators(\s{3,}|\n)"
                    r"\b[IVIX]{1,3}\s*[-\.)]?\s*COST\s*OF\s*THE\s*OPERATION",
                    r"\b[B-F]\s*[-\.)]?\s*(Program|Project)\s*cost\s*and\s*financing",
                    r"\b[B-F]\s*[-\.)]?\sProject\s*outcomes,\s*measurement,\s*monitoring,\s*and\s*evaluation",
                    r"\b[B-F]\s*[-\.)]?\s*Program\s*cost\s*and\s*financing",
                    r"\b[B-F]\s*[-\.)]?\s*Cost\s*and\s*financing",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*(Cost|Costs)\s+and\s+financing",
                    r"\b[B-F]\s*[-\.)]?\s*Project\s*outcomes,\s*measurement,\s*monitoring,\s*and\s*evaluation",
                    r"\b[B-F]\s*[-\.)]?\s*((Expected Results)|(Economic viability)|(Environmental and social risks)|(Fiduciary risks)|(Implementation arrangements))",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[B-F]\s*[-\.)]?\s*Project\s*administration,\s*evaluation,\s*and\s*auditing",  # NEW ADDITIONs @@@@@@@@
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[B-F]\s*[-\.)]?\s*(Results|Result)\s*Matrix\s*with\s*indicators",  # for starting from letters
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[B-F]\s*[-\.)]?\s*Key\s+results\s+indicators",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[B-F]\s*[-\.)]?\s*\n?\s*Key\s+results\s+indicators",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[B-F]\s*[-\.)]?\s+Key\s+results\s+indicators",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[B-F]\s*[-\.)]?\s+Key\s+indicators",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[B-F]\s*[-\.)]?\s*Key\s+results\s+indicators\b",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})(?:\n|$)\b(Key\s+results\s+indicators)\b",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[B-F]\s*[-\.)]?\s*Key\s*Result\s*Indicators",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[B-F]\s*[-\.)]?\s*Expected\s+outcomes\s+of\s+the\s+PROPEF",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[B-F]\s*[-\.)]?\s*Expected\s+outcomes",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[B-F]\s*[-\.)]?\s*Costs\s*and\s*Financing",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[B-F]\s*[-\.)]?\s*Expected\s*results\s*and\s*effectiveness\s*framework",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[B-F]\s*[-\.)]?\s*Financing\s*and\s*cost",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[B-F]\s*[-\.)]?\s*Costs\s*and\s*sources\s*of\s*financing",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[IVIX]{1,3}\s*[-\.)]?\s*COST\s*AND\s*FINANCING",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[B-F]\s*[-\.)]?\s*Key\s+results\s+indicators",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})[B-F]\s*[-\.)]?\s*Key\s+results\s+indicators$",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[B-F]\s*[-\.)]?\s*Costs\s*and\s*Financing",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})((\b[A-F])|(\b[IVIX]{1,3})|(\b\d{1,2}(\.\d{1,2}){0,2})|(\d{1}))\s*[-\.)]?\s*\n?\s*Key\s*results\s*indicators(\s{3,}|\n)"
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[IVIX]{1,3}\s*[-\.)]?\s*(Financial|Associated)\s*(Cost|Costs)",
                    r"\b[B-F]\s*[-\.)]?\s*(Program|Project)\s*cost\s*and\s*financing",
                    r"\b[B-F]\s*[-\.)]?\sProject\s*outcomes,\s*measurement,\s*monitoring,\s*and\s*evaluation",
                    r"\b[B-F]\s*[-\.)]?\s*Program\s*cost\s*and\s*financing",
                    r"\b[B-F]\s*[-\.)]?\s*Cost\s*and\s*financing",
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*Cost\s+and\s+financing",
                    r"\b[B-F]\s*[-\.)]?\s*Project\s*outcomes,\s*measurement,\s*monitoring,\s*and\s*evaluation",
                    r"\b[B-F]\s*[-\.)]?\s*Key\s*indicators\s*in\s*the\s*results\s*matrix",
                    r"\b[B-F]\s*[-\.)]?\s*(Results|Result)\s*(matrix|framework)\s*and\s*key\s*indicators",
                    r"\b[B-F]\s*[-\.)]?\s*Principal\s*outcome\s*indicators",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[IVIX]{1,3}\s*[-\.)]?\s*Key\s*indicators\s*in\s*the\s*results\s*matrix",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[IVIX]{1,3}\s*[-\.)]?\s*(Results|Result)\s*(matrix|framework)\s*and\s*key\s*indicators",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[IVIX]{1,3}\s*[-\.)]?\s*Principal\s*outcome\s*indicators",
                    r"\b[B-F]\s*[-\.)]?\s*((Expected Results)|(Economic viability)|(Environmental and social risks)|(Fiduciary risks)|(Implementation arrangements))",
                    r"\b[IVIX]{1,3}\s*[-\.)]?\s*COST,\s*FINANCING\s*AND\s*EXECUTION\s*TIME",
                    r"\b[IVIX]{1,3}\s*[-\.)]?\s*Project\s*administration,\s*evaluation,\s*and\s*auditing",  # Roman starting (1-9)
                    r"\d{1}\s*[-)\.]\s*Project\s*administration,\s*evaluation,\s*and\s*auditing",  # Single Digit
                    r"\b[IVIX]{1,3}\s*[-\.)]?\s*Program monitoring and evaluation",
                    r"\b[IVIX]{1,3}\s*[-\.)]?\s*(PROGRAM|PROJECT)\s*EXECUTION",
                    r"\b[IVIX]{1,3}\s*[-\.)]?\s*Executing\s*Agency\s*and\s*Execution\s*Structure",
                    r"\b[IVIX]{1,3}\s*[-\.)]?\s*FINANCING\s*STRUCTURE\s*AND\s*MAIN\s*RISKS",
                    r"\b[B-F]\s*[-\.)]?\s*Project\s*outcomes,\s*measurement,\s*monitoring,\s*and\s*evaluation",
                    r"\b[IVIX]{1,3}\s*[-\.)]?\s*Project\s*outcomes,\s*measurement,\s*monitoring",
                    r"\b[IVIX]{1,3}\s*[-\.)]?\s*COST,\s+FINANCING\s+AND\s+EXECUTION\s+TIME",
                    r"V. EXECUTING AGENCY AND EXECUTING MECHANISM",
                    r"VI. ENVIRONMENTAL AND SOCIAL IMPACTS AND PROPOSED ACTIONS",
                    r"Project outcomes, measurement, monitoring, and evaluation",
                    r"\b[B-F]\s*[-\.)]?\s*Outcome\s*indicators",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[IVIX]{1,3}\s*[-\.)]?\s*EXECUTION\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[IVIX]{1,3}\s*[-\.)]?\s*(Executing|Monitoring)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\d{1}\s*[-)\.]\s*n?\s*IMPLEMENTATION\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[IVIX]{1,3}\s*[-\.)]?\s*IMPLEMENTATION\b\s*(?:\n|$)",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*IMPLEMENTATION\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[B-F]\s*[-\.)]?\s*\n?\s*IMPLEMENTATION\b\s*(?:\n|$)"
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(Project\s*administration,\s*evaluation,\s*and\s*auditing)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(Project\s*administration,\s*evaluation,\s*and\s*auditing)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b((Results|Result)\s*Matrix\s*with\s*indicators)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(Key\s+results\s+indicators)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(Key\s*indicators)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(Key\s*Result\s*Indicators)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(Expected\s+outcomes)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(Costs\s*and\s*Financing)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(COST\s*AND\s*FINANCING)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(COST\s*OF\s*THE\s*OPERATION)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b((Program|Project)\s*cost\s*and\s*financing)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(Project\s*outcomes,\s*measurement,\s*monitoring,\s*and\s*evaluation)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(Program\s*cost\s*and\s*financing)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(Project\s*outcomes,\s*measurement,\s*monitoring,\s*and\s*evaluation)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(COST,\s*FINANCING\s*AND\s*EXECUTION\s*TIME)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(Project\s*administration,\s*evaluation,\s*and\s*auditing)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(Project\s*administration,\s*evaluation,\s*and\s*auditing)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b((PROGRAM|PROJECT)\s*EXECUTION)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(Executing\s*Agency\s*and\s*Execution\s*Structure)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(FINANCING\s*STRUCTURE\s*AND\s*MAIN\s*RISKS)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(Project\s*outcomes,\s*measurement,\s*monitoring,\s*and\s*evaluation)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(Project\s*outcomes,\s*measurement,\s*monitoring)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(EXECUTING AGENCY AND EXECUTING MECHANISM)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(ENVIRONMENTAL AND SOCIAL IMPACTS AND PROPOSED ACTIONS)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(Results Framework)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(Results Matrix)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(Project outcomes, measurement, monitoring, and evaluation)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(Outcome\s*indicators)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(Executing|Monitoring)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(IMPLEMENTATION)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(Environmental impact)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(Project benefits )\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(Project feasibility )\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(Project risks)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(Procurement and consulting services)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(Expected results and effectiveness framework)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(Financing and cost)\b\s*(?:\n|$)"
                    r"\b[IVIX]{1,3}\s*[-\.)]?\s*BENEFITS\s*AND\s*RISKS",
                    r"BENEFITS\s*AND\s*RISKS\b\s*(?:\n|$)",
                    r"Key Impacts, Risks, and Mitigation Measures"
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(Operations\s*processing)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(Executing\s*Agencies)\b\s*(?:\n|$)",
                    r"\n+Supervision of the individual operation\b\s*(?:\n|$)",
                    r"\n+Environmental and social considerations\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[IVIX]{1,3}\s*[-\.)]?\s*RATIONALE\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[IVIX]{1,3}\s*[-\.)]?\s*Budget\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[IVIX]{1,3}\s*[-\.)]?\s*EXECUTION\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[IVIX]{1,3}\s*[-\.)]?\s*(Executing|Monitoring)\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[IVIX]{1,3}\s*[-\.)]?\s*IMPLEMENTATION\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[IVIX]{1,3}\s*[-\.)]?\s*COST,\s*FINANCING\s*AND\s*EXECUTION\s*TIME\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[IVIX]{1,3}\s*[-\.)]?\s*FINANCING STRUCTURE AND MAIN RISKS\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[IVIX]{1,3}\s*[-\.)]?\s*FINANCING STRUCTURE AND RISKS\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[IVIX]{1,3}\s*[-\.)]?\s*FINANCING\s*STRUCTURE\s*(?:\n|$)",
                    r"FINANCING STRUCTURE AND MAIN RISKS",
                    r"FINANCING STRUCTURE AND RISKS",
                    r"Results Matrix and key indicators",
                    r"key\s*results\s*indicators",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})(Impact|Results|Result)\s*Indicators",
                    r"Program\s*Execution\s*(?:\n|$)",
                    r"Total cost of the Program",
                    r"cost of the Program",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[IVIX]{1,3}\s*[-\.)]?\s*Procurement\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[IVIX]{1,3}\s*[-\.)]?\s*Financing\s*(?:\n|$)",
                    r"\b[IVIX]{1,3}\s*[-\.)]?\s*BENEFITS\s*AND\s*RISKS",
                    r"BENEFITS\s*AND\s*RISKS\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(Operations\s*processing)\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(Executing\s*Agencies)\b\s*(?:\n|$)",
                    r"\n+Supervision of the individual operation\b\s*(?:\n|$)",
                    r"\n+Environmental and social considerations\b\s*(?:\n|$)",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[IVIX]{1,3}\s*[-\.)]?\s*RATIONALE", #New ones to add
                    r"Key Results Matrix indicators",
                    r"\b[B-F]\s*[-\.)]?\s*Environmental\s*and\s*Social\s*Setting",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b[IVIX]{1,3}\s*[-\.)]?\s*(Financial|Associated)\s*(Cost|Costs)"
                    
                ]
                
                
                project_component_end_2 = [
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})\b(ANNEXES)\b\s*(?:\n|$)",
                    r"\b[IVIX]{1,3}\s*[-\.)]?\s*PROBLEM\s*TO\s*BE\s*ADDRESSED",
                    r"\b[IVIX]{1,3}\s*[-\.)]?\s*(PROJECT|PROJECTS|PROGRAM)\s*DESCRIPTION\s*AND\s*RESULTS\s*MONITORING",
                    r"\b[IVIX]{1,3}\s*[-\.)]?\s*DESCRIPTION\s*AND\s*RESULTS\s*MONITORING",
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*(PROJECT|PROJECTS|PROGRAM)\s*DESCRIPTION\s*AND\s*RESULTS\s*MONITORING(\s{3,}|\n)", #### Perfect pattern for numbers (1.1.1) / 1 / 1.1
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*(PROJECT|PROJECTS|PROGRAM)\s*DESCRIPTION\s*AND\s*RESULTS\s*MONITORING", #### for starting from letters
                    r"\b[IVIX]{1,3}\s*[-\.)]?\s*\n?\s*(PROJECT|PROJECTS|PROGRAM)\s*DESCRIPTION\s*AND\s*RESULTS\s*MONITORING", ### Roman starting (1-9)
                    r"\d{1}\s*[-)\.]\s*n?\s*(PROJECT|PROJECTS|PROGRAM)\s*DESCRIPTION\s*AND\s*RESULTS\s*MONITORING", ### Single Digit
                    r"(\b\d{1,2}(\.\d{1,2}){0,2})\s*[-\.)]?\s*\n?\s*DESCRIPTION\s*AND\s*RESULTS\s*MONITORING(\s{3,}|\n)", #### Perfect pattern for numbers (1.1.1) / 1 / 1.1
                    r"\b[B-F]\s*[-\.)]?\s*\n?\s*DESCRIPTION\s*AND\s*RESULTS\s*MONITORING", #### for starting from letters
                    r"\b[IVIX]{1,3}\s*[-\.)]?\s*\n?\s*DESCRIPTION\s*AND\s*RESULTS\s*MONITORING", ### Roman starting (1-9)
                    r"\d{1}\s*[-)\.]\s*n?\s*DESCRIPTION\s*AND\s*RESULTS\s*MONITORING", ### Single Digit
                    r"\b[IVIX]{1,3}\s*[-\.)]?\s*FRAME\s*OF\s*REFERENCE",
                    r"\b[IVIX]{1,3}\s*[-\.)]?\s*COUNTRY\s*AND\s*PROJECT\s*ELIGIBILITY",
                    r"\b[IVIX]{1,3}\s*[-\.)]?\s*BACKGROUND\s*AND\s*RATIONALE",
                    r"\b[IVIX]{1,3}\s*[-\.)]?\s*Objectives\s*and\s*Justification\s*of\s*the\s*TC",
                    r"\b[IVIX]{1,3}\s*[-\.)]?\s*COUNTRY\s*ELIGIBILITY",
                    r"\b[IVIX]{1,3}\s*[-\.)]?\s*DESCRIPTION\s*OF\s*THE\s*EVENT",
                    r"Objectives\s*and\s*Justification\s*of\s*the\s*TC",
                    r"\b[IVIX]{1,3}\s*[-\.)]?\s*Background",
                    r"LIST OF ACRONYMS",  # "LIST OF ACRONYMS"
                    r"ACRONYMS",  # "ACRONYMS"
                    r"T\s*a\s*b\s*l\s*e\s*\s*o\s*f\s*\s*C\s*o\s*n\s*t\s*e\s*n\s*t",
                    r"\bTable of Contents\b",  # "Table of Contents"
                    r"T\s{1}A\s{1}B\s{1}L\s{1}E",
                    r"F\s{1}R\s{1}A\s{1}M\s{1}E\s{1}W\s{1}O\s{1}R\s{1}K",  # "FRAMEWORK"
                    r"R\s{1}E\s{1}S\s{1}U\s{1}L\s{1}T",  # "RESULT"
                    r"L\s{1}O\s{1}G\s{1}I\s{1}C\s{1}A\s{1}L",  # "LOGICAL"
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})(LOGICAL|Result|Results)\s*FRAMEWORK",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})(LOGICAL|Result|Results)\s*Matrix",
                    r"(\n{1,}|^|(^\s*)|(\n\s*)|\s{2,})DESCRIPTION OF THE PROGRAM",
                    r"FIDUCIARY CONTEXT OF THE COUNTRY",
                    r"FIDUCIARY AGREEMENTS AND REQUIREMENTS",
                    r"(?:\n{2,}[^\S\n]*)ANNEX"  
                    

                ]


                # Identify the Table of Contents page
                toc_page = identify_table_of_contents(pdf_document)
                if toc_page is not None:
                    #print(f"Table of Contents identified on page: {toc_page + 1}")
                    skip_pages = list(range(toc_page + 2))  # Skip TOC page and all pages before it
                else:
                    skip_pages = []

                # Extract text excluding TOC pages (i.e., start from the page after the TOC)
                text = ""
                for page_num in range(pdf_document.page_count):
                    if page_num in skip_pages:  # Skip TOC pages and earlier ones
                        continue
                    text += pdf_document[page_num].get_text("text")
                pdf_document.close()

                
                # Initialize a variable to store the match result
                components_text = None
# Loop through the patterns to find matches
                for pattern in project_component_start:
                    try:
                        # Set the timeout for 5 seconds
                        signal.signal(signal.SIGALRM, timeout_handler)
                        signal.alarm(5)

                        # Attempt to find a match
                        match = re.search(pattern, text, re.IGNORECASE)
                        signal.alarm(0)  # Reset the alarm after the search

                        if match:
                            components_text = match.group(0).strip()
                            component_pattern = pattern
                            print(f"---------\n\nMatch found with pattern:----------- {pattern}\n\n")
                            components_text = clean_text(components_text, project_component_end)
                            components_text = re.sub(r'\n{3,}', '\n\n', components_text)
                            #print(components_text)# Exit the loop if a match is found
                            break
                    except TimeoutException:
                        print(f"Pattern took too long: {pattern}")
                        continue
                    except Exception as e:
                        print(f"An error occurred: {e}")
                        continue
                    finally:
                        signal.alarm(0)#ensure the alarm is reset even if an exception occurs
                
                
                
                all_end = project_component_end + project_component_end_2
                
                #print(components_text)
                
                if components_text and isinstance(components_text, str) and len(components_text.strip()) > 2500:
                    print("\nComponents are long !!!!!!!!!!!!!")
                    components_text = clean_more(components_text, all_end)
                else:
                    print("\nComponents are of good size")
                print(components_text)
                # Fallback if no component text is found
                if not components_text:
                    components_text = "Project Components not found"
                    component_pattern = "Components Pattern N/A"

                extracted_text, summary_info = extract_summary_section(file_path)
                
                if len(extracted_text) > 10000:
                    print(f"\nSummary is long")
                    extracted_text = clean_more(extracted_text, end_keyword)
                    print(extracted_text)
                    
                else:
                    print("Summary section is good")
                
                #print(components_text)
                
                results.append([filename, extracted_text, summary_info, components_text, component_pattern])

        except Exception as e:
            results.append([filename, f"Error processing file: {e}", "Not Available", "Not Available", "Not Available"])
    df = pd.DataFrame(results, columns=["Filename", "Project Summary", "Summary Pattern", "Project Component", "Component Pattern"])
    df = remove_illegal_characters(df)  # Clean the DataFrame
    #df.to_excel(output_excel, index=False)
    #print(f"\n\nResults saved to {output_excel}")        
        
        
path = "/Users/adilqasin/Documents/WBG/Scrapping/ADB/ADB_Downloads/35242-ban-pam.pdf"
output_excel = "/Users/adilqasin/Documents/WBG/Scrapping/IDB/test files/2.xlsx"
process_pdfs_in_folder(path, output_excel)




Processing file: 35242-ban-pam.pdf
Matched Pattern: Contents
---------

Match found with pattern:----------- \b[A-F]\s*[-\.)]?\s*Project\s*(components|component)([\s\S]*)


Pattern used to truncate text:

----------- \b[IVIX]{1,3}\s*[-\.)]?\s*COST\s*ESTIMATES\s*AND\s*FINANCING\s*PLAN

Components are of good size
C. 
Project Components 
4. 
The Project components are as follows: 
 
(i) 
Part A. Gas transmission expansion and reinforcement. Five subcomponents 
will transmit gas to the consumption centers including less developed regions of 
the country. 
 
(a) Part A-1 Ashuganj-Manohardi-Dhanua-Elenga-Jamuna Bridge east bank 
gas transmission pipeline1 (30-inch-diameter, 51 km) and installation of 
compression facilities at Ashuganj (west) and Elenga (AJGTP) 
(b) Part A-2 Hatikumrul-Ishwardi-Bheramara gas transmission pipeline (24-inch-
diameter2, 87 km) (HBGTP).  
(c) Part A-3. Bonpara-Rajshahi gas transmission pipeline (12-inch-diameter, 50 
km) (BRGTP)  
(d) Part A-4. Bheramara-Khuln

/var/folders/pd/vl7v8b496h3fs87b4fnt9qcw0000gn/T/ipykernel_56285/233327365.py:25: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(clean_cell)


In [63]:
r"((\b[A-F])|(\b[IVIX]{1,3})|(\b\d{1,2}(\.\d{1,2}){0,2})|(\d{1}))\s*[-\.)]?\s*\n?\s*HEADING(\s{3,}|\n)" ###COMPLETE HEADING

'((\\b[A-F])|(\\b[IVIX]{1,3})|(\\b\\d{1,2}(\\.\\d{1,2}){0,2})|(\\d{1}))\\s*[-\\.)]?\\s*\\n?\\s*HEADING(\\s{3,}|\\n)'

In [34]:
import re

pattern = r"five\s*components\s*(.*)"
text = "some content here five components something after"

match = re.search(pattern, text)
if match:
    print("Match found:", match.group(1))
else:
    print("No match found.")


Match found: something after


In [126]:
print(all_end)

NameError: name 'all_end' is not defined